# Part 2: Preprocessing & Feature Engineering
## From SMILES to Enriched Graphs

This notebook preprocesses raw ChEMBL data and creates multiple feature representations:
- **Morgan Fingerprints** (2048-bit): Circular substructure patterns
- **MACCS Keys** (166-bit): Structural patterns
- **Molecular Graphs** (32-dim): Atom-level features
- **Enriched Graphs** (2246-dim): Graph + fingerprint injection

In [ ]:
# @title 1. Setup & Data Loading
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from pathlib import Path

# Load data
df = pd.read_csv('data/chembl_vegfr2.csv')
print(f"Raw data: {len(df)} compounds")
df.head()

In [ ]:
# @title 2. Preprocess: Validate, Deduplicate, Label
from vegfr2.data import preprocess, split

# Use vegfr2 preprocessing pipeline
df_clean = preprocess(df)
print(f"After preprocessing: {len(df_clean)} compounds")
print(f"  Active: {df_clean['active'].sum()} ({df_clean['active'].mean():.1%})")
print(f"  Inactive: {(1-df_clean['active']).sum()} ({1-df_clean['active'].mean():.1%})")

In [ ]:
# @title 3. Train/Val/Test Split
train_df, val_df, test_df = split(df_clean, seed=42)

print(f"Split sizes:")
print(f"  Train: {len(train_df)} ({train_df['active'].mean():.1%} active)")
print(f"  Val:   {len(val_df)} ({val_df['active'].mean():.1%} active)")
print(f"  Test:  {len(test_df)} ({test_df['active'].mean():.1%} active)")

# Save splits
train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv', index=False)
test_df.to_csv('data/test.csv', index=False)

In [ ]:
# @title 4. Morgan Fingerprints (2048-bit)
from vegfr2.features import smiles_to_morgan

print("Extracting Morgan fingerprints (radius=2, 2048 bits)...")
X_train_morgan = np.vstack([smiles_to_morgan(s) for s in train_df['smiles']])
X_val_morgan = np.vstack([smiles_to_morgan(s) for s in val_df['smiles']])
X_test_morgan = np.vstack([smiles_to_morgan(s) for s in test_df['smiles']])

print(f"  Shape: {X_train_morgan.shape}")
print(f"  Sample bits: {X_train_morgan[0, :20]}")

In [ ]:
# @title 5. MACCS Keys (166-bit)
from vegfr2.features import smiles_to_maccs

print("Extracting MACCS structural keys (166 bits)...")
X_train_maccs = np.vstack([smiles_to_maccs(s) for s in train_df['smiles']])
X_val_maccs = np.vstack([smiles_to_maccs(s) for s in val_df['smiles']])
X_test_maccs = np.vstack([smiles_to_maccs(s) for s in test_df['smiles']])

print(f"  Shape: {X_train_maccs.shape}")

# Combined features
X_train_both = np.hstack([X_train_morgan, X_train_maccs])
X_val_both = np.hstack([X_val_morgan, X_val_maccs])
X_test_both = np.hstack([X_test_morgan, X_test_maccs])

print(f"  Combined (Morgan+MACCS): {X_train_both.shape[1]}-dim")

In [ ]:
# @title 6. Molecular Graph Construction
from vegfr2.features import mol_to_graph

# Example: single molecule
smiles_example = train_df['smiles'].iloc[0]
graph = mol_to_graph(smiles_example)

print(f"Example molecule: {smiles_example}")
print(f"  Node features: {graph['node_feats'].shape} (32-dim per atom)")
print(f"  Edge index: {graph['edge_index'].shape} (2, num_edges)")
print(f"  Edge features: {graph['edge_feats'].shape} (11-dim per bond)")
print(f"  Num atoms: {graph['num_nodes']}")

In [ ]:
# @title 7. Enriched Graph Construction (2246-dim)
from vegfr2.features import mol_to_graph_with_fps, get_enriched_node_dim

print("Building enriched graphs: [atom(32) + Morgan(2048) + MACCS(166)] = 2246-dim per node")

enriched = mol_to_graph_with_fps(smiles_example, use_morgan=True, use_maccs=True)
enriched_dim = get_enriched_node_dim(use_morgan=True, use_maccs=True)

print(f"\nExample: {smiles_example}")
print(f"  Enriched node features: {enriched['node_feats'].shape}")
print(f"  Expected dim: {enriched_dim}")
print(f"  Each atom gets FULL molecular fingerprint appended")

In [ ]:
# @title 8. Feature Dimension Summary
print("=" * 60)
print("FEATURE REPRESENTATION SUMMARY")
print("=" * 60)
print(f"\n{'Feature Type':<30} {'Dimension':>10}")
print("-" * 42)
print(f"{'Morgan Fingerprint':<30} {'2048':>10}")
print(f"{'MACCS Keys':<30} {'166':>10}")
print(f"{'Morgan + MACCS':<30} {'2214':>10}")
print(f"{'Atom Features (graph)':<30} {'32':>10}")
print(f"{'Enriched Node (graph+FP)':<30} {'2246':>10}")
print(f"{'Bond Features':<30} {'11':>10}")
print("=" * 60)

In [ ]:
# @title 9. Visualization
import matplotlib.pyplot as plt
from vegfr2.features import mol_to_graph_with_fps

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Morgan fingerprint sparsity
axes[0].imshow(X_train_morgan[:50, :100], aspect='auto', cmap='binary')
axes[0].set_title('Morgan FP (50 molecules, first 100 bits)')
axes[0].set_xlabel('Bit index')
axes[0].set_ylabel('Molecule')

# MACCS keys
axes[1].imshow(X_train_maccs[:50, :], aspect='auto', cmap='binary')
axes[1].set_title('MACCS Keys (50 molecules, 166 bits)')
axes[1].set_xlabel('Key index')
axes[1].set_ylabel('Molecule')

# Enriched features
enriched_sample = np.vstack([mol_to_graph_with_fps(s)['node_feats'].mean(dim=0).numpy() 
                              for s in train_df['smiles'].iloc[:50]])
axes[2].imshow(enriched_sample[:, :100], aspect='auto', cmap='viridis')
axes[2].set_title('Enriched Node Features (50 molecules, first 100 dims)')
axes[2].set_xlabel('Feature dimension')
axes[2].set_ylabel('Molecule')

plt.tight_layout()
plt.savefig('images/feature_visualization.png', dpi=150, bbox_inches='tight')
plt.show()